<a href="https://colab.research.google.com/github/comp0161/tutorials/blob/main/comp0161_2026_lab3_sonification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP0161 Auditory Computing Week 3: Sonification

As part of this week's tutorial session, we will perform a simple **sonification** — producing an audio representation of a dataset. There are many possible ways to do this, and I make no claims that the one presented here is even *good*, let alone the best. But it will hopefully serve as an illustration of the concept.

As in last week's tutorial, we will be using very basic sound generation functions -- many of the same ones, in fact. Much richer and more capable systems exist, but they tend to be quite complex, with extensive and potentially unintuitive APIs. The simpler approach taken here is much less fully featured but should hopefully be somewhat comprehensible.

# Setting Up

We'll mostly be using packages that are installed by default on Colab, but we'll want to add a bit of reverb later on so let's install pedalboard again.

In [ ]:
%pip install --quiet pedalboard

Import libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from IPython.display import Image, Audio

# we'll be wanting some random numbers later
# we create a generator with a fixed seed for consistent behaviour
# if you want different results, change the seed
SEED = 9907
rng = np.random.default_rng(SEED)

import pedalboard

# Data

Sonification is very dependent on data. What works for one kind of data will be inappropriate for another. In this case we're going to make use of some climate time series data, showing **temperature anomalies** (differences from some baseline) over the period 1850—2025.

<details>
<summary>Note</summary>
This data is from the US <a href="https://www.ncei.noaa.gov/products/land-based-station/noaa-global-temp">National Oceanic and Atmospheric Administration</a>. It's freely available at the time of writing but might well not be by the time you read this, NOAA being one of many US government agencies in the firing line of the current political administration.
</details>

In [ ]:
!curl -O https://comp0161.github.io/assets/data/anomalies.csv

Let's load it up and have a look.

In [ ]:
df = pd.read_csv("anomalies.csv")

In [ ]:
# the date is encoded as YYYYMM, which is a bit irregular
# split and remap so it spaces uniformly
df['Year'] = df['date'] // 100
df['Month'] = df['date'] % 100
df['Year_frac'] = df['Year'] + ((df['Month'] - 1) * 1/12)

In [ ]:
plt.plot(df['Year_frac'], df['north'], label='North')
plt.plot(df['Year_frac'], df['south'], label='South')
plt.plot(df['Year_frac'], df['global'], label='Overall')
plt.xlabel("Year")
plt.ylabel("Temperature Anomaly (°C)")
plt.legend();

# Sounds

Our basic sound synthesis functions are mostly the same as last week's. We'll skip the narrative since you've seen it all before.

In [ ]:
PITCH = 440
PITCHES = [55, 110, 220, 440, 880]
DURATION = 1
SAMPLE_RATE = 44100

# this time we'll include some phase randomisation by default
# for reasons that will be discussed below
RAND_PHASE = 1

def play(x, rate=SAMPLE_RATE, dupe_stereo=True):
    """
    Display a numpy array using IPython.Audio.
    Optionally duplicates mono to stereo (on by default).
    """
    if dupe_stereo and (len(x.shape) == 1):
        x = np.stack((x,x)).copy()

    display(Audio(x, rate=rate))

def tone(hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, phase=0, random_phase=RAND_PHASE):
    """
    Generate a simple pure sine tone of specified frequency
    and duration.
    """
    return np.sin(rng.uniform() * random_phase * 2 * np.pi + phase + hz * 2 * np.pi * np.arange(int(duration * rate))/rate)

def tones(hz=PITCHES, duration=DURATION, weights=None, rate=SAMPLE_RATE, phases=None):
    """
    Generate a (potentially weighted) sum of pure sine tones.
    """
    if weights is None:
        weights = np.ones(len(hz))

    if phases is None:
        phases = np.zeros(len(hz))

    result = weights[0] * tone(hz[0], duration, rate, phases[0])
    for ii in range(1, len(hz)):
        result += weights[ii] * tone(hz[ii], duration, rate, phases[ii])

    return result

def saw (hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, max_harm=10000, random_phase=RAND_PHASE ):
    """
    Generate a band-limited sawtooth wave of specified frequency
    and duration. Phases of the individual harmonics can be randomised
    by an amount specified by`random_phase` (interpreted as
    largest fraction of a cycle).
    """
    assert(hz < rate/2)

    N = int(duration * SAMPLE_RATE)

    angles = 2 * np.pi * np.arange(N)/rate
    if max_harm is None: max_harm = 10000
    max_harm = np.min([int(np.floor(rate / (2 * hz))), max_harm])

    # start with the fundamental
    result = -np.sin(angles * hz)
    components = 1

    # add the harmonics
    for harm in range(2, max_harm + 1):
        result -= np.sin(angles * hz * harm + rng.uniform() * random_phase * 2 * np.pi ) / harm
        components += 1

    # scale into [-1, 1]
    result = 2 * (result - np.min(result))/(np.max(result) - np.min(result)) - 1

    return result

def square (hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, max_harm=10000, random_phase=RAND_PHASE ):
    """
    Generate a band-limited square wave of specified frequency
    and duration. Phases of the individual harmonics can be randomised
    by an amount specified by`random_phase` (interpreted as
    largest fraction of a cycle).
    """
    assert(hz < rate/2)

    N = int(duration * SAMPLE_RATE)

    angles = 2 * np.pi * np.arange(N)/rate

    if max_harm is None: max_harm = 10000
    max_harm = np.min([int(np.floor(rate / (2 * hz))), max_harm])

    # start with the fundamental
    result = np.sin(angles * hz)

    components = 1

    # add the harmonics
    for harm in range(3, max_harm + 1, 2):
        result += np.sin(angles * hz * harm + rng.uniform() * random_phase * 2 * np.pi ) / harm
        components += 1

    # scale into [-1, 1]
    result = 2 * (result - np.min(result))/(np.max(result) - np.min(result)) - 1

    return result

def noise(duration=DURATION, rate=SAMPLE_RATE, shape=None):
    """
    Generate N noise samples with the power spectrum
    optionally shaped by the supplied function.
    """
    N = int(duration * SAMPLE_RATE)
    noise_spectrum = np.fft.rfft(rng.standard_normal(N))

    if shape is not None:
        shape_spectrum = shape(np.fft.rfftfreq(N))
        # normalise to preserve energy
        shape_spectrum /= np.sqrt(np.mean(shape_spectrum**2))

        noise_spectrum *= shape_spectrum

    return np.fft.irfft(noise_spectrum);

BLUE = lambda x: np.sqrt(x)
VIOLET = lambda x: x
BROWN = lambda x: 1/np.where(x==0, np.inf, x)
PINK = lambda x: 1/np.where(x==0, np.inf, np.sqrt(x))

def envelope(duration=DURATION, rate=SAMPLE_RATE,
             attack=0.01, decay=0.05, sustain=0, release=0):
    """
    Simple linear ADSR envelope. Attack, decay and sustain are
    specified in seconds, generated according to the rate. Sustain
    fills everything not taken up by the other phases. If the
    duration is insufficient to contain all the elements the
    excess is discarded.
    """
    N = int(duration * rate)
    result = np.full(N, sustain, dtype=float)

    nR = np.min((int(release * rate), N))
    if nR:
        result[(N - nR):] = np.linspace(sustain, 0, nR)

    nA = int(attack * rate)
    if nA:
        result[:nA] = np.linspace(0, 1, nA)

    nD = np.min((int(decay * rate), N-nA))
    if nD:
        result[nA:(nA + nD)] = np.linspace(1, result[np.min((N, nA+nD))], nD)

    return result

We'll use a different kind of envelope -- a raised cosine function or **Hann window** -- to make little snippets of sound that we can add together to build up a sequence.

In [ ]:
def hann(duration=DURATION, rate=SAMPLE_RATE):
  """
  Generate Hann window.
  """
  N = int(duration * rate)
  tt = np.linspace(0, 2 * np.pi, N)
  return (1 - np.cos(tt))/2

In a slight abuse of terminology I'm going to call these snippets **grains**. This method is vaguely similar to [granular synthesis](https://en.wikipedia.org/wiki/Granular_synthesis), but that usually uses recorded samples as the source material.

In [ ]:
def grain(func=tone, hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, random_phase=RAND_PHASE):
  """
  Generate a single windowed snippet of some waveform.
  """
  return func(hz=hz, duration=duration, rate=rate, random_phase=random_phase) * hann(duration, rate)

def grains(freqs, func=tone, duration=DURATION, rate=SAMPLE_RATE, random_phase=RAND_PHASE):
  """
  Generate a sound stream by summing a sequence of windowed snippets
  of the specified frequencies.
  """
  N = int(duration * rate)
  count = len(freqs)
  half_step = int(np.ceil(N/2))
  result = np.zeros((count + 1) * half_step)
  for ii in range(count):
    result[(ii * half_step):(ii * half_step + N)] += grain(func, freqs[ii], duration, rate, random_phase)

  return result

Here's a quick test to check that it works, building a simple frequency sweep. Note that the fixed grain spacing produces banding artefacts. Phase randomisation mitigates these a bit, though at cost of introducing a different kind of artefact -- less regular but more textural.

In [ ]:
play(grains(np.arange(50,1500,5), func=tone, duration=0.1, random_phase=0))
play(grains(np.arange(50,1500,5), func=tone, duration=0.1, random_phase=1))

We'll also do some simple stereo spatialisation. We haven't covered this in the lectures yet -- we'll get into it more in week 7 -- but I'm sure you're familiar with the basic concept.

In [ ]:
def mono(x):
  """
  Flatten a multichannel sound to mono by summing all channels.
  """
  assert(len(x.shape)==2)
  return np.sum(x, axis=0)


def stereo(x, pan=0.5):
  """
  Generate a stereo sound from mono by adjusting
  the level on each side.

  `pan` goes from 0 (fully left) to 1 (fully right).

  (Out of range pan values will also work but may produce strange effects.)
  """
  assert(len(x.shape)==1)
  return np.stack((x * 1-pan, x * pan))

# Sonification

Let's review the data we want to sonify:

In [ ]:
years = df['Year_frac'][df.shape[0]-1] - df['Year_frac'][0]
print(f'{df.shape[0]} samples covering {years} years')
print(f'Global data range: {np.min(df["global"])} – {np.max(df["global"])}')
print(f'Northern data range: {np.min(df["north"])} – {np.max(df["north"])}')
print(f'Southern data range: {np.min(df["south"])} – {np.max(df["south"])}')

We have 2101 data points, spanning 175 years. For the moment, we'll arbitrarily use 2 seconds per decade, for an overall sonification duration of about 35 seconds.

<details>
<summary>Note</summary>
There are a lot of arbitrary decisions here. I'm presenting it as if there's some clear rationale, and there sort of is, but also quite a bit of trial and error along the way. You should expect that in your own sonifications too.
</details>

In [ ]:
# grains half overlap, so each grain spans two timesteps
# ie, this is 2 * SECONDS / (YEARS * MONTHS)
grain_duration = 2 * 2/(10 * 12)

We have three parallel time series, closely related but non-identical. We're going to use a couple of different strategies to represent these sonically.

We'll start by constructing a varying pitch sequence for each one. These will all share the same timescale, so that the events are in sync, but we'll represent them differently in space and pitch.

In [ ]:
# in each case we'll use the data to vary the frequency around some baseline
# we'll exponentiate because frequency is perceived logarithmically
# also, we want frequency to be non-negative
global_base = 55
global_scaled = np.exp(2 * df['global'].values)
global_pitches = global_base * global_scaled

plt.plot(global_pitches)
global_audio = grains(global_pitches, func=tone, duration=grain_duration)
play(global_audio)

In [ ]:
north_base = 440
north_scaled = np.exp(df['north'].values)
north_pitches = north_base * north_scaled

plt.plot(north_pitches)
north_audio = grains(north_pitches, func=tone, duration=grain_duration)
play(north_audio)

In [ ]:
south_base = 440
south_scaled = np.exp(df['south'].values)
south_pitches = south_base * south_scaled

plt.plot(south_pitches)
south_audio = grains(south_pitches, func=tone, duration=grain_duration)
play(south_audio)

We can put these together with some stereo positioning: north to the left, south to the right, global in the middle.

In [ ]:
combined = stereo(global_audio) + stereo(north_audio, 0) + stereo(south_audio, 1)
play(combined)

As a sonic representation of the data, this is fine as far as it goes. It gives a sense of the underlying rising trend with a fair bit of local volatility on top. But it might be interesting to pick it apart a bit more and represent some of the components more distinctly.

For example, we could smooth out the trend with a rolling median filter, and then subtract that to look at the transients.

In [ ]:
med_window = 24
global_meds = np.array([np.median(df['global'].values[ii:ii+med_window]) for ii in range(df.shape[0]-med_window)])
global_meds = np.concatenate((np.full(med_window//2, global_meds[0]), global_meds, np.full(med_window//2, global_meds[-1])))

plt.plot(df['Year_frac'], df['global'])
plt.plot(df['Year_frac'], global_meds);

In [ ]:
north_meds = np.array([np.median(df['north'].values[ii:ii+med_window]) for ii in range(df.shape[0]-med_window)])
north_meds = np.concatenate((np.full(med_window//2, north_meds[0]), north_meds, np.full(med_window//2, north_meds[-1])))

plt.plot(df['Year_frac'], df['north'])
plt.plot(df['Year_frac'], north_meds);

In [ ]:
south_meds = np.array([np.median(df['south'].values[ii:ii+med_window]) for ii in range(df.shape[0]-med_window)])
south_meds = np.concatenate((np.full(med_window//2, south_meds[0]), south_meds, np.full(med_window//2, south_meds[-1])))

plt.plot(df['Year_frac'], df['south'])
plt.plot(df['Year_frac'], south_meds);

In [ ]:
plt.plot(df['Year_frac'], global_meds, label='global')
plt.plot(df['Year_frac'], north_meds, label='north')
plt.plot(df['Year_frac'], south_meds, label='south')
plt.legend()

The filtered versions are still not exactly smooth, but they're a lot less spiky. Let's build the same kind of pitch sequences as before. However, this time I'm going to shift the global line up a few octaves, leaving the bass register clear for later.

In [ ]:
global_base = 880
global_scaled = np.exp(2 * global_meds)
global_pitches = global_base * global_scaled
global_audio = grains(global_pitches, func=tone, duration=grain_duration)
play(global_audio)

north_base = 440
north_scaled = np.exp(2 * north_meds)
north_pitches = north_base * north_scaled
north_audio = grains(north_pitches, func=tone, duration=grain_duration)
play(north_audio)

south_base = 440
south_scaled = np.exp(2 * south_meds)
south_pitches = south_base * south_scaled
south_audio = grains(south_pitches, func=tone, duration=grain_duration)
play(south_audio)

combined = stereo(global_audio) + stereo(north_audio, 0) + stereo(south_audio, 1)
play(combined)

Now that we've got the smoothed lines out of the way, let's check out the residual noise.

In [ ]:
plt.plot(df['Year_frac'], df['north'].values - north_meds)
plt.plot(df['Year_frac'], df['global'].values - global_meds)
plt.plot(df['Year_frac'], df['south'].values - south_meds)

This seems like it might make for some interesting (and sinister!) bass rumbles, so let's build those pitch sequences too.

In [ ]:
rumble_base = 55
global_noise = np.exp(2 * (df['global'].values - global_meds))
global_rumble = grains(rumble_base * global_noise, func=tone, duration=grain_duration)
play(global_rumble)

north_noise = np.exp(2 * (df['north'].values - north_meds))
north_rumble = grains(1.5 * rumble_base * north_noise, func=tone, duration=grain_duration)
play(north_rumble)

south_noise = np.exp(2 * (df['south'].values - south_meds))
south_rumble = grains(1.5 * rumble_base * south_noise, func=tone, duration=grain_duration)
play(south_rumble)

combined_rumble = stereo(global_rumble) + stereo(north_rumble, 0) + stereo(south_rumble, 1)
play(combined_rumble)

In [ ]:
play(combined + combined_rumble)

So far we've considered only properties that change more or less continuously over time, making for an eerie but not necessarily very interpretable soundscape. But we might also want to pick out specific discrete events and represent them with single transient sounds.

Let's take another look at our noise sequences.

In [ ]:
plt.plot(df['Year_frac'], north_noise)
plt.plot(df['Year_frac'], global_noise)
plt.plot(df['Year_frac'], south_noise)

The noise rumbles along the whole time, but there are definitely some distinct peaks, so let's make a feature of those.

<details>
<summary>Note</summary>
In this case these peaks are not necessarily very meaningful — there will often be outliers in noisy data that don't really signify much beyond the fact that noise is noise. But we're really just illustrating the idea here.
</details>

We'll attach some simple one-off sounds at each peak in the top percentile.

In [ ]:
global_peaks = np.flatnonzero(global_noise > np.percentile(global_noise, 99))
north_peaks = np.flatnonzero(north_noise > np.percentile(north_noise, 99))
south_peaks = np.flatnonzero(south_noise > np.percentile(south_noise, 99))

GLOBAL_TING = envelope() * saw(2200)
NORTH_TING = envelope() * square(2000)
SOUTH_TING = envelope() * square(1800)
play(GLOBAL_TING)
play(NORTH_TING)
play(SOUTH_TING)

Indexing these to the right times is slightly fiddly, but there's nothing too complicated going on: because the grains overlap, each data point represents a step forward in time of half the grain duration.

In [ ]:
global_tings = np.zeros_like(global_audio)
for idx in global_peaks:
  offset = int(idx * SAMPLE_RATE * grain_duration / 2)
  endpoint = min(offset+len(GLOBAL_TING), len(global_tings))
  global_tings[offset:endpoint] += GLOBAL_TING[:endpoint-offset]
play(global_tings)

north_tings = np.zeros_like(north_audio)
for idx in north_peaks:
  offset = int(idx * SAMPLE_RATE * grain_duration / 2)
  endpoint = min(offset+len(NORTH_TING), len(north_tings))
  north_tings[offset:endpoint] += NORTH_TING[:endpoint-offset]
play(north_tings)

south_tings = np.zeros_like(south_audio)
for idx in south_peaks:
  offset = int(idx * SAMPLE_RATE * grain_duration / 2)
  endpoint = min(offset+len(SOUTH_TING), len(south_tings))
  south_tings[offset:endpoint] += SOUTH_TING[:endpoint-offset]
play(south_tings)

combined_tings = stereo(global_tings) + stereo(north_tings, 0) + stereo(south_tings, 1)
play(combined_tings)

Ok, let's add these to the rumbles and trend lines, weighting the tings up a bit so they don't get too lost in the mix.

(While we're at it let's add a bit of silence at the end so the sound doesn't just stop abruptly. That will also leave room for the reverb tails in the next step.)

In [ ]:
mix = np.concatenate((2 * combined + combined_rumble + 4 * combined_tings, np.zeros((2,4*SAMPLE_RATE))), axis=1)
play(mix)

Finally, just for the aesthetic value let's throw in a bit of reverb. As noted last week, everything's better with reverb!

In [ ]:
reverb = pedalboard.Reverb(room_size=0.75, damping=0.5, wet_level=0.5, dry_level=0.5)
play(reverb.process(mix, SAMPLE_RATE))

Voilà!

This is by no means the only or best way to sonify this or any other data. We've made a lot of arbitrary choices just for the sake of demonstration. It's not clear we've actually made anything in the dataset more intuitive or accessible in this case. Feel free to experiment yourself and try to do better!